In [1]:
import os, sys, json, re, time, subprocess
from pathlib import Path

try:
    import pandas as pd
    import numpy as np
except Exception:
    import sys as _s, subprocess as _sp
    _sp.check_call([_s.executable, '-m', 'pip', 'install', '-q', 'pandas', 'numpy', 'scikit-learn', 'mlflow', 'fastapi', 'uvicorn', 'joblib', 'requests'])
    import pandas as pd
    import numpy as np

p = Path.cwd()
while not (p / 'churnDataset.csv').exists() and p.parent != p:
    p = p.parent
os.chdir(p)
rt = Path.cwd()
dd = rt / 'data'
md = rt / 'models'
ld = rt / 'logs'
mlr = rt / 'mlruns'
for d in [dd, md, ld]:
    d.mkdir(exist_ok=True)

import joblib
import mlflow
import mlflow.sklearn
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.impute import KNNImputer, SimpleImputer

def cn(x):
    x = str(x).strip().lower()
    x = re.sub(r'[^a-z0-9]+', '_', x)
    return x.strip('_')

def cl(df):
    df = df.copy()
    df.columns = [cn(c) for c in df.columns]
    df = df.drop_duplicates()
    for c in df.select_dtypes(include='object').columns:
        df[c] = df[c].astype(str).str.strip().replace({'': np.nan, 'nan': np.nan, 'None': np.nan})
    df['churn'] = pd.to_numeric(df['churn'], errors='coerce')
    return df

def fe(df):
    df = df.copy()
    df['spend_per_tenure'] = df['total_spend'] / df['tenure'].replace(0, np.nan)
    df['call_delay_ratio'] = df['support_calls'] / (df['payment_delay'] + 1)
    df['usage_recent_score'] = df['usage_frequency'] / (df['last_interaction'] + 1)
    df['spend_usage_ratio'] = df['total_spend'] / (df['usage_frequency'] + 1)
    return df.replace([np.inf, -np.inf], np.nan)

def miss_a(df):
    df = df.copy()
    ns = df.select_dtypes(include=np.number).columns
    cs = [c for c in df.columns if c not in ns]
    for c in ns:
        df[c] = df[c].fillna(df[c].median())
    for c in cs:
        m = df[c].mode(dropna=True)
        df[c] = df[c].fillna(m.iloc[0] if len(m) else 'unknown')
    return df

def miss_b(df):
    df = df.copy()
    ns = list(df.select_dtypes(include=np.number).columns)
    cs = [c for c in df.columns if c not in ns]
    if ns:
        df[ns] = KNNImputer(n_neighbors=5).fit_transform(df[ns])
    if cs:
        df[cs] = SimpleImputer(strategy='most_frequent').fit_transform(df[cs])
    return df

def out(df):
    df = df.copy()
    ns = [c for c in df.select_dtypes(include=np.number).columns if c not in ['churn', 'customerid', 'customer_id']]
    mk = pd.Series(True, index=df.index)
    for c in ns:
        q1 = df[c].quantile(0.25)
        q3 = df[c].quantile(0.75)
        iq = q3 - q1
        if iq > 0:
            mk &= df[c].between(q1 - 1.5 * iq, q3 + 1.5 * iq)
    rs = df.loc[mk].reset_index(drop=True)
    return rs if len(rs) else df.reset_index(drop=True)

def run_data(method='knn'):
    df = pd.read_csv('churnDataset.csv')
    df = fe(cl(df))
    a = out(miss_a(df))
    b = out(miss_b(df))
    a.to_csv(dd / 'processed_median_mode.csv', index=False)
    b.to_csv(dd / 'processed_knn_mode.csv', index=False)
    tb = b if method == 'knn' else a
    tb['churn'] = tb['churn'].round().astype(int)
    fs = [c for c in tb.columns if c not in ['churn', 'customerid', 'customer_id']]
    ns = [c for c in fs if pd.api.types.is_numeric_dtype(tb[c])]
    cs = [c for c in fs if c not in ns]
    st = {c: {'mean': float(tb[c].mean()), 'std': float(tb[c].std() or 1.0)} for c in ns}
    tb.to_csv(dd / 'processed_churn.csv', index=False)
    (dd / 'columns.json').write_text(json.dumps({'features': fs, 'num': ns, 'cat': cs}, indent=2), encoding='utf-8')
    (dd / 'train_stats.json').write_text(json.dumps(st, indent=2), encoding='utf-8')
    print('data ok', tb.shape[0], tb.shape[1])
    return tb, fs, ns, cs

def oh():
    try:
        return OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown='ignore', sparse=False)

def ev(m, x, y):
    pr = m.predict(x)
    return {'accuracy': float(accuracy_score(y, pr)), 'f1': float(f1_score(y, pr))}

fp = dd / 'processed_churn.csv'
if not fp.exists():
    tb, fs, ns, cs = run_data()
else:
    tb = pd.read_csv(fp)
    cm = json.loads((dd / 'columns.json').read_text(encoding='utf-8'))
    fs, ns, cs = cm['features'], cm['num'], cm['cat']

mlflow.set_tracking_uri(mlr.resolve().as_uri())
mlflow.set_experiment('churn')
x = tb[fs]
y = tb['churn'].astype(int)
xtr, xt, ytr, yt = train_test_split(x, y, test_size=0.3, stratify=y, random_state=42)
xv, xte, yv, yte = train_test_split(xt, yt, test_size=0.5, stratify=yt, random_state=42)
ms = {
    'lr': LogisticRegression(max_iter=1000, class_weight='balanced'),
    'rf': RandomForestClassifier(n_estimators=150, random_state=42, class_weight='balanced', n_jobs=-1),
}
rs = []
for nm, al in ms.items():
    with mlflow.start_run(run_name=nm) as rn:
        pp = ColumnTransformer([('n', StandardScaler(), ns), ('c', oh(), cs)])
        m = Pipeline([('pp', pp), ('md', al)])
        m.fit(xtr, ytr)
        va = ev(m, xv, yv)
        te = ev(m, xte, yte)
        pm = al.get_params()
        mlflow.log_params({k: v for k, v in pm.items() if isinstance(v, (str, int, float, bool, type(None)))})
        mlflow.log_metric('val_accuracy', va['accuracy'])
        mlflow.log_metric('val_f1', va['f1'])
        mlflow.log_metric('test_accuracy', te['accuracy'])
        mlflow.log_metric('test_f1', te['f1'])
        mlflow.sklearn.log_model(m, 'model')
        rs.append({'name': nm, 'run_id': rn.info.run_id, 'val_accuracy': va['accuracy'], 'val_f1': va['f1'], 'test_accuracy': te['accuracy'], 'test_f1': te['f1']})
        print(nm, round(va['accuracy'], 4), round(va['f1'], 4), round(te['accuracy'], 4), round(te['f1'], 4))

best = max(rs, key=lambda r: r['val_f1'])
bm = mlflow.sklearn.load_model(f"runs:/{best['run_id']}/model")
joblib.dump(bm, md / 'best_model.pkl')
(md / 'runs.json').write_text(json.dumps(rs, indent=2), encoding='utf-8')
(md / 'best_run.json').write_text(json.dumps(best, indent=2), encoding='utf-8')
(md / 'sample_input.json').write_text(json.dumps(x.iloc[0].to_dict(), indent=2), encoding='utf-8')
print('best', best['name'], round(best['val_f1'], 4))

subprocess.Popen(['mlflow', 'ui', '--backend-store-uri', 'mlruns', '--host', '0.0.0.0', '--port', '5000'])
time.sleep(3)
try:
    from google.colab import output
    output.serve_kernel_port_as_window(5000)
except Exception:
    print('MLflow UI: http://127.0.0.1:5000')
rs


C:\Users\Admin\AppData\Local\Programs\Python\Python310\lib\site-packages\requests\__init__.py:109: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.4.3)/charset_normalizer (3.4.1) doesn't match a supported version!
  warnings.warn(


2026/05/04 13:28:21 INFO mlflow.tracking.fluent: Experiment with name 'churn' does not exist. Creating a new experiment.


2026/05/04 13:28:21 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


2026/05/04 13:28:47 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


lr 0.8615 0.8697 0.8588 0.8668


...

2026/05/04 13:28:52 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


rf 0.9973 0.9974 0.997 0.9971


best rf 0.9974


MLflow UI: http://127.0.0.1:5000


[{'name': 'lr',
  'run_id': '2ce8b2ad473e4e6990692658e1f241f8',
  'val_accuracy': 0.8615113672750561,
  'val_f1': 0.8697092935683085,
  'test_accuracy': 0.8587896253602305,
  'test_f1': 0.8668478260869565},
 {'name': 'rf',
  'run_id': 'fa0cc00f9aa44cf99da40e8c1ce663fb',
  'val_accuracy': 0.9972782580851746,
  'val_f1': 0.9974269713939761,
  'test_accuracy': 0.9969580531540185,
  'test_f1': 0.9971251323952186}]